# B3b Defense – 07 LLM Macroeconomic Analysis

**Objective:** Send the XGBoost forecast results (`defense_forecast_2026_sac.csv`) and
the model driver breakdown (`defense_forecast_2026_drivers_sac.csv`) to Claude Sonnet 4.6
and request a macroeconomic interpretation tailored to a company planning to enter the
US defense segment with new products in 2026.

**Target variable:** FDEFX — Federal Government National Defense Consumption Expenditures
and Gross Investment (USD billions, SAAR). This is the primary forecast output; ADEFNO
(new defense orders) is a model input feature, not the forecasted quantity.

The LLM acts as a macroeconomic analyst writing for a corporate controller — it does not
re-run any model, but contextualises the quantitative outputs for strategic planning and
management communication.

In [16]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from IPython.display import display, Markdown

# Force UTF-8 output on Windows (avoids CP1252 encoding errors)
if sys.stdout.encoding and sys.stdout.encoding.lower() != 'utf-8':
    sys.stdout.reconfigure(encoding='utf-8')

# Load API key from project .env
load_dotenv(dotenv_path='../../.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        'ANTHROPIC_API_KEY not found. Add it to the project .env file: '
        'ANTHROPIC_API_KEY=sk-ant-...'
    )

client         = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'
OUTPUT_DIR     = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Anthropic client initialised')

Anthropic client initialised


In [17]:
# ===========================================================================
# Load DoD P-1 FY2026 procurement data and compute branch/subcategory summaries
# ===========================================================================
p1_raw = pd.read_excel(DATA_RAW + 'Procurement_Programs_P1.xlsx',
                       sheet_name='FY 2026 Total', header=1)

# Filter to "Add" lines only (Non-Add rows are informational memo entries)
p1_add = p1_raw[p1_raw['Add/Non-Add'] == 'Add'].copy()
p1_add['FY 2026 Total Amount'] = pd.to_numeric(
    p1_add['FY 2026 Total Amount'], errors='coerce'
)


def _map_branch(org):
    """Map P-1 organization codes to service branches."""
    if org == 'A':
        return 'Army'
    elif org == 'N':
        return 'Navy/Marine Corps'
    elif org == 'F':
        return 'Air Force/Space Force'
    else:
        return 'Defense-Wide'


p1_add['Branch'] = p1_add['Organization'].apply(_map_branch)

# Branch totals in USD billions (source amounts are in thousands USD → /1e6)
branch_totals = (
    p1_add.groupby('Branch')['FY 2026 Total Amount']
    .sum()
    .sort_values(ascending=False)
    / 1e6
)
p1_total_bn = round(branch_totals.sum(), 1)

branch_pct = branch_totals / branch_totals.sum() * 100
p1_branch_lines = '\n'.join(
    f'  - {branch}: ${val:.1f} bn ({branch_pct[branch]:.1f}%)'
    for branch, val in branch_totals.items()
)

# Top subcategories from "Other Procurement" and "Shipbuilding" accounts
op_ship = p1_add[
    p1_add['Account Title'].str.contains(
        'Other Procurement|Shipbuilding', case=False, na=False
    )
].copy()

# Take head(7) before filtering so that 6 non-classified entries remain
relevant_sub = (
    op_ship.groupby('Budget SubActivity (BSA) Title')['FY 2026 Total Amount']
    .sum()
    .sort_values(ascending=False)
    .head(7)
    / 1e6
)

# Remove classified entries (not useful as qualitative context for the LLM)
relevant_sub = relevant_sub[
    ~relevant_sub.index.get_level_values('Budget SubActivity (BSA) Title')
    .str.contains('Classified', case=False, na=False)
]
relevant_sub = relevant_sub.head(6)

p1_relevant_lines = '\n'.join(
    f'  {i + 1}. {name}: ${val:.1f} bn'
    for i, (name, val) in enumerate(relevant_sub.items())
)

print(f'P-1 FY2026 total (Add lines): ${p1_total_bn:.1f} bn')
print('Branch breakdown:')
print(p1_branch_lines)
print()
print('Top subcategories (Other Procurement + Shipbuilding, excl. Classified):')
print(p1_relevant_lines)

P-1 FY2026 total (Add lines): $223.8 bn
Branch breakdown:
  - Navy/Marine Corps: $103.4 bn (46.2%)
  - Air Force/Space Force: $46.4 bn (20.7%)
  - Defense-Wide: $43.0 bn (19.2%)
  - Army: $31.1 bn (13.9%)

Top subcategories (Other Procurement + Shipbuilding, excl. Classified):
  1. Other Warships: $24.2 bn
  2. Fleet Ballistic Missile Ships: $9.3 bn
  3. Auxiliaries, Craft and Prior Yr Program Cost: $7.1 bn
  4. Amphibious Ships: $4.5 bn
  5. Reactor Plant Equipment: $2.9 bn
  6. Other Shipboard Equipment: $2.8 bn


In [18]:
# ===========================================================================
# 1. Load FDEFX statistics — PRIMARY FORECAST TARGET
# ===========================================================================
fdefx_raw = pd.read_csv(DATA_RAW + 'FDEFX.csv', parse_dates=['observation_date'])
fdefx_raw = fdefx_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

fdefx_hist     = fdefx_raw[fdefx_raw['date'] <= '2025-12-31'].copy()
fdefx_last_row = fdefx_hist.iloc[-1]
fdefx_last_val = fdefx_last_row['FDEFX']
fdefx_last_qtr = fdefx_last_row['date']

q_num = (fdefx_last_qtr.month - 1) // 3 + 1
fdefx_last_quarter = f'Q{q_num} {fdefx_last_qtr.year}'

prior_date      = fdefx_last_qtr - pd.DateOffset(years=1)
fdefx_prior_row = fdefx_hist[fdefx_hist['date'] == prior_date]
if fdefx_prior_row.empty:
    fdefx_prior_row = fdefx_hist.iloc[-5:-4]
fdefx_prior_val = float(fdefx_prior_row['FDEFX'].iloc[0])
fdefx_yoy       = (fdefx_last_val - fdefx_prior_val) / fdefx_prior_val * 100

# 5-year range for context
fdefx_2020_2025 = fdefx_hist[fdefx_hist['date'].dt.year >= 2020]['FDEFX']
fdefx_min_val   = fdefx_2020_2025.min()
fdefx_max_val   = fdefx_2020_2025.max()

# ===========================================================================
# 2. Load ADEFNO statistics — context feature (defense new orders)
# ===========================================================================
adefno_raw = pd.read_csv(DATA_RAW + 'ADEFNO.csv', parse_dates=['observation_date'])
adefno_raw = adefno_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

adefno_hist       = adefno_raw[adefno_raw['date'] <= '2025-12-31'].copy()
last_row          = adefno_hist.iloc[-1]
adefno_last_month = last_row['date'].strftime('%b %Y')
adefno_last_value = last_row['ADEFNO']

adefno_2025    = adefno_hist[adefno_hist['date'].dt.year == 2025]['ADEFNO']
adefno_12m_avg = adefno_2025.mean()
adefno_vs_avg_pct = (adefno_last_value / adefno_12m_avg - 1) * 100

# ===========================================================================
# 3. Load IPB52300S statistics — defense production index (context feature)
# ===========================================================================
ipb_raw = pd.read_csv(DATA_RAW + 'IPB52300S.csv', parse_dates=['observation_date'])
ipb_raw = ipb_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

ipb_hist             = ipb_raw[ipb_raw['date'] <= '2025-12-31'].copy()
ipb_last_row         = ipb_hist.iloc[-1]
ipb_last_val         = ipb_last_row['IPB52300S']
ipb_last_month_label = ipb_last_row['date'].strftime('%b %Y')

ipb_jan_val    = ipb_hist[ipb_hist['date'] == '2025-01-01']['IPB52300S'].iloc[0]
ipb_12m_change = ipb_last_val - ipb_jan_val
ipb_direction  = 'upward' if ipb_12m_change > 0 else 'downward'

# ===========================================================================
# 4. Load FDEFX forecast and compute summary statistics
# ===========================================================================
forecast_df = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_sac.csv')
forecast_df['date_parsed'] = pd.to_datetime(
    forecast_df['Date'].astype(str), format='%Y%m'
)
forecast_df['Month']      = forecast_df['date_parsed'].dt.strftime('%b %Y')
# FDEFX is in USD bn SAAR — convert from full USD back to billions for display
forecast_df['FDEFX_USD_bn'] = (forecast_df['Net_Value_USD'] / 1e9).round(1)

forecast_avg_bn = round(forecast_df['Net_Value_USD'].mean() / 1e9, 1)
forecast_min_bn = round(forecast_df['Net_Value_USD'].min() / 1e9, 1)
forecast_max_bn = round(forecast_df['Net_Value_USD'].max() / 1e9, 1)
fdefx_step_down = round(fdefx_last_val - forecast_avg_bn, 1)

jan_2026_val = round(
    forecast_df[forecast_df['date_parsed'].dt.month == 1]['Net_Value_USD'].values[0] / 1e9, 1
)

forecast_table = forecast_df[['Month', 'FDEFX_USD_bn']].to_string(index=False)

# ===========================================================================
# 5. Load model driver breakdown (SHAP)
# ===========================================================================
drivers_df   = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_drivers_sac.csv')
feature_shap = drivers_df[~drivers_df['Driver'].isin(['base_value', 'prediction'])].copy()
feature_shap['abs_shap'] = feature_shap['SHAP_Value_monthly_USD_bn'].abs()

base_value_bn = float(
    drivers_df[drivers_df['Driver'] == 'base_value']['SHAP_Value_monthly_USD_bn'].iloc[0]
)

importance_df = (
    feature_shap.groupby('Driver')['abs_shap']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'Driver': 'Feature', 'abs_shap': 'Avg_contribution_USD_bn'})
)
importance_df['Avg_contribution_USD_bn'] = importance_df['Avg_contribution_USD_bn'].round(2)
importance_table = importance_df.to_string(index=False)

# Individual and cumulative driver contributions for prompt narrative
def _get_contrib(feature):
    row = importance_df[importance_df['Feature'] == feature]['Avg_contribution_USD_bn']
    return float(row.values[0]) if len(row) > 0 else 0.0

lag1_contrib       = _get_contrib('fdefx_lag_1')
lag2_contrib       = _get_contrib('fdefx_lag_2')
lag3_contrib       = _get_contrib('fdefx_lag_3')
ipb_contrib_val    = _get_contrib('IPB52300S')
adefno_contrib_val = _get_contrib('ADEFNO_lag_6')
cumul_lag123       = round(lag1_contrib + lag2_contrib + lag3_contrib, 2)
cumul_external     = round(ipb_contrib_val + adefno_contrib_val, 2)

print(f'FDEFX last quarter:    {fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn SAAR (YoY {fdefx_yoy:+.1f}%)')
print(f'FDEFX 2020–2025 range: {fdefx_min_val:,.1f} – {fdefx_max_val:,.1f} USD bn SAAR')
print(f'ADEFNO last value:     {adefno_last_month}: {adefno_last_value:,.0f} USD mn ({adefno_vs_avg_pct:+.1f}% vs 2025 avg)')
print(f'ADEFNO 12m avg 2025:   {adefno_12m_avg:,.0f} USD mn')
print(f'IPB52300S last:        {ipb_last_month_label}: {ipb_last_val:.2f} (12m change: {ipb_12m_change:+.2f}, {ipb_direction})')
print(f'Forecast 2026:         Jan {jan_2026_val} → avg {forecast_avg_bn} → range {forecast_min_bn}–{forecast_max_bn} USD bn SAAR')
print(f'Step-down from Q4 act: {fdefx_step_down:.1f} USD bn')
print(f'Model reference level: {base_value_bn:.1f} USD bn SAAR (SHAP base value)')
print(f'Cumul. lag-1/2/3:      {cumul_lag123:.2f} USD bn  |  External signals: {cumul_external:.2f} USD bn')

FDEFX last quarter:    Q4 2025: 1,159.2 USD bn SAAR (YoY +3.2%)
FDEFX 2020–2025 range: 872.0 – 1,161.9 USD bn SAAR
ADEFNO last value:     Dec 2025: 18,774 USD mn (+19.0% vs 2025 avg)
ADEFNO 12m avg 2025:   15,780 USD mn
IPB52300S last:        Dec 2025: 112.96 (12m change: +3.48, upward)
Forecast 2026:         Jan 75.5 → avg 75.1 → range 74.8–75.5 USD bn SAAR
Step-down from Q4 act: 1084.1 USD bn
Model reference level: 61.1 USD bn SAAR (SHAP base value)
Cumul. lag-1/2/3:      10.20 USD bn  |  External signals: 0.53 USD bn


## Prompt Design

The system prompt positions Claude as a senior macroeconomic analyst writing for a
**corporate controller** — technical ML terms (XGBoost, SHAP, autoregressive) are avoided
in the output; business-language equivalents are used instead.

**Target variable:** FDEFX (realized US federal defense expenditures, USD bn SAAR).
ADEFNO (new defense orders) and IPB52300S (production index) are model input features,
not the forecasted quantity.

**Anti-hallucination rules injected into the user prompt:**
- Use **only** the numerical values explicitly provided (no memory-based statistics)
- Every factual claim not derivable from the tables must cite an **official primary source**
  with a verifiable URL (FRED, BEA, US Census, Federal Reserve, CBO, DoD Comptroller)
- No secondary sources; no industry association claims without a direct primary URL
- If a claim cannot be supported → **omit it entirely**
- Scenario probabilities flagged as **illustrative** unless sourced

**Output structure (4 sections + exec summary):**
1. Defense Market Volume & Outlook (FDEFX trajectory, step-down from 2025, stabilisation)
2. What Drives the Forecast (plain-language: recent spending momentum, policy signal, production capacity)
3. Quarterly Monitoring KPIs (Markdown table: 4 KPIs with source URL, cadence, what to watch)
4. Key Risks & Forecast Limitations (US budget dynamics, frozen macro inputs)
5. Executive Summary (exactly 6 bullets, max 2 lines each — for SAC Planning Story)

In [19]:
# ---------------------------------------------------------------------------
# Model selection — uncomment the model you want to use
# ---------------------------------------------------------------------------
# claude-sonnet-4-5   $3.00 input / $15.00 output per 1M tokens
# MODEL = 'claude-sonnet-4-5'

# claude-haiku-4-5    $1.00 input / $5.00 output  per 1M tokens  (cheapest)
# MODEL = 'claude-haiku-4-5'

# claude-sonnet-4-6   $3.00 input / $15.00 output per 1M tokens  (latest)
MODEL = 'claude-sonnet-4-6'

# claude-opus-4-6     $5.00 input / $25.00 output per 1M tokens  (most capable)
# MODEL = 'claude-opus-4-6'
# ---------------------------------------------------------------------------

MAX_TOKENS = 3500

# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are a trusted senior advisor writing for a corporate controller who is sharp on
numbers but new to the US defense market. Your tone is warm and collegial — like a
knowledgeable colleague briefing them over coffee, not a consultant billing by the hour.
Never talk down. When defense-specific terms are unavoidable, explain them naturally in
the same sentence as if it is obvious context, not a lecture.
Your job: give the controller a clear picture of the US defense market — who spends,
on what, and what that means strategically for a company entering the US defense segment
in 2026 with zero prior defense revenue (industrial B2B: compressors, pneumatic tools).
Rules: data explanations max one sentence each; spend the majority on market context,
segment dynamics, and strategic implications; use only numbers explicitly provided or
cite official primary sources (FRED, BEA, CBO, DoD Comptroller) with URL; no
geopolitical speculation; no modelling jargon; be concise — quality over length.
Plain language rule: every sentence must be understandable to a controller with no
defense market background. Forbidden without an inline plain-language explanation:
CR, P-1, FDEFX, ADEFNO, IPB52300S, AVL, DLA, NSN, DFARS, NAVSEA, FSC, RDT&E, O&M,
MRO, FY, BEA, CBO. Explain each naturally in the flow of the sentence — no brackets
full of jargon, no footnotes.
Formatting rule: never write prose paragraphs. Every section uses a bold lead sentence
followed by bullet points. Each bullet maximum 15 words. The reader should be able to
scan the full document in under 90 seconds.\
"""

USER_PROMPT = f"""\
## Input Data

Forecast: {forecast_avg_bn} USD bn/month for 2026 (~{round(forecast_avg_bn*12,0):.0f} USD bn annualised). Last actual ({fdefx_last_quarter}): {fdefx_last_val:,.1f} USD bn annualised ({fdefx_yoy:+.1f}% YoY). Source: https://fred.stlouisfed.org/series/FDEFX

Leading indicators ({adefno_last_month}): ADEFNO {adefno_last_value:,.0f} USD mn ({adefno_vs_avg_pct:.0f}% above 2025 avg of {adefno_12m_avg:,.0f} USD mn, https://fred.stlouisfed.org/series/ADEFNO). IPB52300S {ipb_last_val:.1f} (+{ipb_12m_change:.1f} pts over 2025, https://fred.stlouisfed.org/series/IPB52300S).

DoD FY2026 Procurement P-1 (https://comptroller.defense.gov/Budget-Materials/), ${p1_total_bn:.0f}B Add lines, not comparable to FDEFX:
{p1_branch_lines}
Top subcategories (Other Procurement + Shipbuilding, excl. Classified):
{p1_relevant_lines}

## Output — six sections in this exact order, separated by --- between each

### Executive Summary
5 bullets, max 20 words each. Start each bullet with a bold status tag in this exact order:
- **[SIZE]** market size
- **[SEGMENT]** most relevant segment
- **[ENTRY]** entry point
- **[RISK]** top risk
- **[ACTION]** next action
The [SIZE] bullet must reference the 2026 forecast level, not the Q4 2025 actual.

---

### Section 1 — Market Outlook
Write one bold lead sentence that captures the key point, followed by 3–5 bullet points
of max 15 words each. No paragraph prose.
Cover: what FDEFX (US federal defense spending, measured as actual money flowing to
suppliers) measures and why it matters here; the 2026 forecast level; budget dynamics,
continuing resolution risk, what the elevated new-orders signal means. Cite CBO or DoD
Comptroller where relevant.

---

### Section 2 — Where the Money Flows
Write one bold lead sentence that captures the key point, followed by 3–5 bullet points
of max 15 words each. No paragraph prose.
Cover: which branches dominate and why; which shipbuilding and other-procurement
subcategories are relevant for compressors/pneumatic tools; customer concentration risk
from branch mix. No numerical comparison of P-1 procurement figures to FDEFX.

---

### Section 2b — Forecast Driver Decomposition
Write one bold lead sentence that captures the key point, followed by 3–5 bullet points
of max 15 words each. No paragraph prose.
Cover: which DoD spending categories — procurement (buying new equipment), operations
and maintenance (running and servicing existing equipment), and research and development
— and which service branches primarily drive the ~{round(forecast_avg_bn*12,0):.0f} USD bn
annualised forecast level. Explain specifically why the forecast sits at THIS level in
2026, not just where money flows in general. Identify which categories or branches are
growing vs. contracting and what that implies for the addressable market for compressors
and pneumatic tools.
Include exactly one bridge sentence in this form: "The forecast of
~{round(forecast_avg_bn*12,0):.0f} USD bn is primarily driven by [spending category or
branch], because [budget dynamic]."
After the bullet points, add a two-column Markdown table with exactly 3 rows:
| Signal | What this means for us |
Draw the 3 rows directly from the bullet point content of this section.
Cite DoD Comptroller (https://comptroller.defense.gov) or CBO (https://www.cbo.gov)
where relevant.

---

### Section 3 — Strategic Implications
Write one bold lead sentence that captures the key point, followed by 3–5 bullet points
of max 15 words each. No paragraph prose.
Cover: which segment to prioritise first and why; likely procurement channels; top risks
before committing to a market share assumption.
If a term requires inline explanation, put the explanation in a separate bullet directly
below — do not extend the parent bullet beyond 15 words.

---

### Section 4 — 4 KPIs
Markdown table: KPI | Source URL | Cadence | What to watch for. Focus on market entry
thesis, not just macro movement. No other changes to this section.
"""

# ---------------------------------------------------------------------------
# Print first 200 characters of USER_PROMPT for verification (no API call)
# ---------------------------------------------------------------------------
print('USER_PROMPT first 200 characters:')
print(repr(USER_PROMPT[:200]))
print()

# ---------------------------------------------------------------------------
# Print full prompt before API call
# ---------------------------------------------------------------------------
SEP = '=' * 72
print(SEP)
print('SYSTEM PROMPT')
print(SEP)
print(SYSTEM_PROMPT)
print()
print(SEP)
print('USER PROMPT')
print(SEP)
print(USER_PROMPT)
print(SEP)
print(f'Model:         {MODEL}')
print(f'Max tokens:    {MAX_TOKENS}')
print(f'Prompt length: {len(USER_PROMPT):,} characters')
print(SEP)
print()

# ---------------------------------------------------------------------------
# API call (streaming)
# ---------------------------------------------------------------------------
print(f'Calling {MODEL} ...\n')
print('-' * 72)

llm_response = ''
with client.messages.stream(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': USER_PROMPT}],
) as stream:
    for text in stream.text_stream:
        llm_response += text
        print(text, end='', flush=True)

print('\n' + '-' * 72)
print('Streaming complete.')

USER_PROMPT first 200 characters:
'## Input Data\n\nForecast: 75.1 USD bn/month for 2026 (~901 USD bn annualised). Last actual (Q4 2025): 1,159.2 USD bn annualised (+3.2% YoY). Source: https://fred.stlouisfed.org/series/FDEFX\n\nLeading in'

SYSTEM PROMPT
You are a trusted senior advisor writing for a corporate controller who is sharp on
numbers but new to the US defense market. Your tone is warm and collegial — like a
knowledgeable colleague briefing them over coffee, not a consultant billing by the hour.
Never talk down. When defense-specific terms are unavoidable, explain them naturally in
the same sentence as if it is obvious context, not a lecture.
Your job: give the controller a clear picture of the US defense market — who spends,
on what, and what that means strategically for a company entering the US defense segment
in 2026 with zero prior defense revenue (industrial B2B: compressors, pneumatic tools).
Rules: data explanations max one sentence each; spend the majority on market

In [20]:
# Render response as formatted Markdown in the notebook
display(Markdown(llm_response))

# Save to data/processed — overwrite existing file, no date suffix
output_path = Path(DATA_PROCESSED) / 'lllm_defense_analysis_2026.md'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('# LLM Macroeconomic Analysis — US Defense Market 2026\n\n')
    f.write(f'**Model:** {MODEL}  \n')
    f.write(f'**Prompt length:** {len(USER_PROMPT):,} characters  \n\n')
    f.write('---\n\n')
    f.write(llm_response)

print(f'Response saved to: {output_path}')

## Executive Summary

- **[SIZE]** US federal defense spending forecast ~$901 bn annualized for 2026, a significant market.
- **[SEGMENT]** Navy shipbuilding dominates; compressors and pneumatic tools serve ship construction and maintenance directly.
- **[ENTRY]** Target Navy ship-systems suppliers as indirect customers; direct DoD sales require approved vendor status.
- **[RISK]** Continuing resolution — Congress funding government on a temporary, no-new-contracts basis — delays procurement awards.
- **[ACTION]** Register in SAM.gov and pursue approved-vendor qualification with a Navy prime contractor immediately.

---

### Section 1 — Market Outlook

**Defense spending is large, growing, and signaling a strong 2026 pipeline — but timing risk is real.**

- Federal defense spending (actual dollars flowing to suppliers, per FRED series FDEFX) hit $1,159 bn annualized in Q4 2025.
- The 2026 forecast steps down to ~$901 bn annualized, reflecting normal budget-cycle timing, not a policy cut.
- New defense orders (FRED series ADEFNO) in December 2025 ran 19% above the 2025 average — a strong forward signal.
- Industrial production for defense (FRED series IPB52300S) rose 3.5 points over 2025, confirming real output growth.
- A continuing resolution — a short-term stopgap that funds government without approving new contracts — remains a key timing risk for 2026 awards.

---

### Section 2 — Where the Money Flows

**Navy and Marine Corps dominate procurement, and their shipbuilding programs are the most direct fit for compressors and pneumatic tools.**

- Navy/Marine Corps holds 46% of the DoD's fiscal year 2026 procurement budget — the single largest share by far.
- Warship construction and Fleet Ballistic Missile submarines are the top subcategories — both require compressed-air and pneumatic systems.
- "Other Shipboard Equipment" and "Auxiliaries and Craft" are smaller but highly relevant for industrial tooling and maintenance gear.
- Army procurement is the smallest branch share, and its programs skew toward vehicles and munitions — lower fit.
- Heavy Navy concentration means our addressable market is not spread evenly; one customer segment dominates.

---

### Section 2b — Forecast Driver Decomposition

**The ~$901 bn forecast reflects operations and maintenance dominance, not a procurement surge — which shapes where compressors fit.**

- Operations and maintenance — the budget category covering day-to-day running, repair, and servicing of existing military equipment — is the largest single DoD spending category, per DoD Comptroller ([comptroller.defense.gov](https://comptroller.defense.gov)).
- Procurement — buying new ships, aircraft, and weapons — is a smaller but fast-growing share, driven by Navy shipbuilding expansion.
- Research and development spending is growing but is largely software and electronics-focused; low direct relevance for pneumatic tools.
- The forecast of ~$901 bn is primarily driven by operations and maintenance spending, because the existing fleet is large and aging, requiring sustained servicing budgets even when new procurement is delayed.
- Navy procurement is growing; Army and Air Force operations budgets are relatively flat, per CBO's latest baseline ([cbo.gov](https://www.cbo.gov)).

| Signal | What this means for us |
|---|---|
| Operations & maintenance is the largest and most stable category | MRO — maintenance, repair, and overhaul of existing equipment — is our most accessible near-term revenue path |
| Navy procurement is the fastest-growing segment | New shipbuilding programs create durable, multi-year demand for compressors and pneumatic assembly tools |
| Army/Air Force budgets are flat or slow-growing | Prioritizing Navy over other branches is not just strategic preference — it follows the money |

---

### Section 3 — Strategic Implications

**Enter through Navy shipbuilding primes; direct DoD sales are a second-phase objective, not a starting point.**

- Navy prime contractors — large integrators who hold the main shipbuilding contracts — are the realistic first customer.
- Qualifying as an approved vendor on a prime's supply chain is faster than pursuing direct government contracts.
- The Defense Logistics Agency — the DoD's central purchasing body for industrial supplies — is a viable direct channel, but requires catalog registration and compliance steps.
- *(Plain language: "approved vendor" means the prime contractor has audited and accepted you as a qualified supplier.)*
- Procurement channels require compliance with defense acquisition regulations, adding cost and lead time before first revenue.
- *(Plain language: defense acquisition regulations — DFARS — are a set of government purchasing rules layered on top of standard commercial contracts.)*
- Top risks: a continuing resolution freezing new awards, long qualification timelines, and competition from incumbents already on approved lists.

---

### Section 4 — 4 KPIs

| KPI | Source URL | Cadence | What to watch for |
|---|---|---|---|
| US Federal Defense Spending (actual dollars flowing to suppliers) | [https://fred.stlouisfed.org/series/FDEFX](https://fred.stlouisfed.org/series/FDEFX) | Quarterly | Sustained spend above $900 bn annualized confirms market entry timing is sound |
| New Defense Orders (forward demand signal from manufacturers) | [https://fred.stlouisfed.org/series/ADEFNO](https://fred.stlouisfed.org/series/ADEFNO) | Monthly | Orders staying above 2025 average signal pipeline health for 12–18 months out |
| Defense Industrial Production Index (actual output at defense suppliers) | [https://fred.stlouisfed.org/series/IPB52300S](https://fred.stlouisfed.org/series/IPB52300S) | Monthly | Rising index means primes are actively building — driving near-term tooling and compressed-air demand |
| DoD Procurement Budget (line-item budget request by branch and program) | [https://comptroller.defense.gov/Budget-Materials/](https://comptroller.defense.gov/Budget-Materials/) | Annual (February) | Watch Navy shipbuilding line growth; flat or cut lines signal reduced new-build opportunity |

Response saved to: ..\data\processed\lllm_defense_analysis_2026.md
